### Import Dependencies

In [ ]:
from pydantic import BaseModel

import cohere

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

from langsmith import traceable, get_current_run_tree

import instructor

from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.types import Send

from langchain_core.messages import SystemMessage, convert_to_openai_messages, HumanMessage, AIMessage
from IPython.display import Image, display

from typing import Literal, Dict, Any, Annotated, List
from pydantic import Field
from operator import add

import random
import openai
import pandas as pd

from jinja2 import Template


from qdrant_client import QdrantClient
from qdrant_client import models
from qdrant_client.models import VectorParams, Distance, SparseVectorParams, Modifier, PayloadSchemaType, PointStruct, Document, Prefetch, FusionQuery 

### Agent Graph with Loopback from Tools (ReAct Agent)

In [ ]:
@traceable(
        name="embed_query",
        run_type="embedding",
        metadata={
            "ls_provider": "openai",
            "ls_model_name": "text-embedding-3-small"
        }
)
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )

    current_run = get_current_run_tree()
    if current_run:
        current_run.metadata["usage_metadata"] = {
            "input_tokens": response.usage.prompt_tokens,
            "total_tokens": response.usage.total_tokens,
        }

    return response.data[0].embedding


@traceable(
        name="retrieve_data",
        run_type="retriever"
)
def retrieve_data(query, qdrant_client, k=5, hybrid=True):

    query_embedding = get_embedding(query)

    if hybrid:
        results = qdrant_client.query_points(
            collection_name="Amazon-items-collection-01-hybrid-search",
            prefetch=[
                Prefetch(
                    query=query_embedding,
                    using="text-embedding-3-small",
                    limit=20
                ),
                Prefetch(
                    query=Document(
                        text=query,
                        model="qdrant/bm25"
                    ),
                    using="bm25",
                    limit=20
                )
            ],
            query=models.RrfQuery(rrf=models.Rrf(weights=[3,1])),
            limit=k
        )
    else:
        results = qdrant_client.query_points(
            collection_name="Amazon-items-collection-01-hybrid-search",
            query=query_embedding,
            using="text-embedding-3-small",
            limit=k
        )

    retrieved_context_ids = []
    retrieved_contexts = []
    similarity_scores = []
    retrieved_contexts_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_contexts.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_contexts_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_contexts": retrieved_contexts,
        "similarity_scores": similarity_scores,
        "retrieved_contexts_ratings": retrieved_contexts_ratings
    }


@traceable(
        name="rerank_data",
        run_type="tool"
)
def rerank_data(query, context, top_k=5):

    cohere_client = cohere.ClientV2()

    response = cohere_client.rerank(
        model="rerank-v4.0-pro",
        query=query,
        documents=context["retrieved_contexts"],
        top_n=top_k
    )

    order = [result.index for result in response.results]

    return {
        "retrieved_context_ids": [context["retrieved_context_ids"][i] for i in order],
        "retrieved_contexts": [context["retrieved_contexts"][i] for i in order],
        "similarity_scores": [context["similarity_scores"][i] for i in order],
        "retrieved_contexts_ratings": [context["retrieved_contexts_ratings"][i] for i in order]
    }


@traceable(
        name="format_retrieved_conext",
        run_type="prompt"
)
def process_context(context):
    
    formated_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_contexts"], context["retrieved_contexts_ratings"]):
        formated_context += f"- ID: {id},  rating: {rating},  description: {chunk}\n"

    return formated_context


@tool
def get_formatted_item_context(query: str, top_k: int = 5) -> str:

    """Search available products and return the top k matching inventory items.

Expand the customer's question into 1-5 concise search statements and issue them
in parallel in a single turn. Each statement covers one distinct product or
attribute; no two may express the same intent. Use natural product-description
language. If no brand or model is specified, search broadly rather than refusing.

    "Earphones for me and a waterproof speaker"
        -> "Personal earphones" | "Waterproof speaker"
    "A warm winter jacket for hiking"
        -> "Insulated winter jacket" | "Hiking outerwear for cold weather"

Before calling, check what earlier calls in this conversation already returned.
Search only for what is missing; results already retrieved remain valid and must
not be fetched again.

Args:
    query: A single search statement describing one product or attribute.
    top_k: Number of items to retrieve. Works best with 5 or more.

Returns:
    A string of the top k available products, each prefixed with its ID and
    average rating.
    """

    qdrant_client = QdrantClient(url="http://localhost:6333")

    retrieved_context = retrieve_data(
        query,
        qdrant_client,
        k=20
    )

    retrieved_context = rerank_data(query, retrieved_context, top_k=top_k)
    formatted_context = process_context(retrieved_context)

    return formatted_context



### State and Pydantic Models for Structured Outputs

In [ ]:
class RAGUsedContext(BaseModel):
    id: str = Field("ID of the item used to answer the question")
    description: str = Field(description="Description of the item used to answer the question")

class FinalResponse(BaseModel):

    """Call this tool when the final answer is possible using available context."""

    answer: str = Field(description="Answer to the question")
    references: list[RAGUsedContext] = Field(description="List of items used to answer the question")

class State(BaseModel):
    messages: Annotated[List[Any], add] = []
    question_relevant: bool = False
    iteration: int = 0
    answer: str = ""
    final_answer: bool = False
    references: list[RAGUsedContext] = []

In [ ]:
@traceable(
    name="agent_node",
    run_type="llm",
    metadata={
        "ls_provider": "openai",
        "ls_model_name": "gpt-5.4-mini"
    }
)
def agent_node(state: State) -> dict:

    prompt_template = """You are a shopping assistant that answers customer questions about products currently in stock.

## Procedure

Before every tool call, check what previous tool calls in this conversation already
returned. Search only for what's genuinely missing. If nothing is missing, call
FinalResponse instead — previously retrieved data is as valid as fresh data, and
re-running a search you already ran is an error.

Customer: "Which of those speakers is cheapest?"
→ Speakers and prices already retrieved. No search. FinalResponse.

Customer: "Does that jacket come with a rain shell, and do you have gloves?"
→ Jacket specs already retrieved; gloves are not. Search gloves only.

## Answering

- Never state a product detail that isn't in the retrieved data.
- Describe products with specifications in bullet points.
- In references, include every chunk that contributed to your answer with the chunk id and product name.
- Call retrieved data "available products", never "context".
- Nothing relevant returned → say so, ask the customer to refine.
- Off-topic question → ask what product they're interested in.
"""

    template = Template(prompt_template)

    prompt = template.render()

    llm = ChatOpenAI(
        model="gpt-5.4-mini",
        reasoning_effort="low",
        use_responses_api=True
    )
    llm_with_tools = llm.bind_tools(
        [get_formatted_item_context, FinalResponse],
        tool_choice="required"
    )

    response = llm_with_tools.invoke(
        [
            SystemMessage(content=prompt),
            *state.messages
        ]
    )

    final_answer = False
    answer = ""
    references = []

    def sanitize_response(response):

        for tool_call in response.tool_calls:
            if tool_call.get("name") == "FinalResponse":
                answer = tool_call.get("args").get("answer")

        return AIMessage(content=answer)


    if len(response.tool_calls) > 0:
        for tool_call in response.tool_calls:
            if tool_call.get("name") == "FinalResponse":
                final_answer = True
                answer = tool_call.get("args").get("answer")
                references.extend(tool_call.get("args").get("references"))

                response = sanitize_response(response);

    return {
        "messages": [response],
        "final_answer": final_answer,
        "iteration": state.iteration + 1, 
        "answer": answer,
        "references": references
    }

In [ ]:
def tool_router(state: State) -> str:

    if state.final_answer:
        return "end"
    elif state.iteration > 2:
        return "end"
    elif len(state.messages[-1].tool_calls) > 0:
        return "tools"
    else:
        return "end"

## User Intent Router Node

In [ ]:
class IntentRouterResponse(BaseModel):
    question_relevant: bool
    answer: str = Field(description="An answer to the question if the users question is not relevant to the products.")

In [ ]:
@traceable(
        name="route_intent",
        run_type="llm",
        metada={
            "ls_provider": "openai",
            "ls_model_name": "gpt-5.4-mini"
        }
)
def intent_router_node(state: State) -> dict:

    prompt_template = """You are a relevance router for a shopping assistant that answers questions about products in stock.

## Instructions

- Determine whether the question is about products, inventory, or purchasing.
- Questions about product features, availability, pricing, comparisons, and recommendations are relevant.
- Questions about store policies, personal advice, or unrelated topics are not relevant.

## Examples

Question: "Do you have running shoes under $100?"
Relevant: yes

Question: "What's the weather like today?"
Relevant: no - not related to products

Question: "Can you help me write an essay?"
Relevant: no - not related to products

Question: "Which laptop has the best battery life?"
Relevant: yes

Question: "What's your return policy?"
Relevant: no - about store policy, not product information
"""

    template = Template(prompt_template)

    prompt = template.render()

    messages = state.messages

    conversation = []

    for message in messages:
        conversation.append(convert_to_openai_messages(messages[-1]))

    client = instructor.from_provider(
        "openai/gpt-5.4-mini",
        mode=instructor.Mode.RESPONSES_TOOLS
    )

    response, raw_response = client.create_with_completion(
            messages=[
            {"role": "system", "content": prompt},
            *conversation
        ],
        reasoning={"effort": "none"},
        response_model=IntentRouterResponse
    )

    return {
        "question_relevant": response.question_relevant,
        "answer": response.answer
    }

In [ ]:
def intent_router_conditional_edges(state: State) -> str:

    if state.question_relevant:
        return "agent_node"
    else:
        return "end"

### Graph Construction

In [ ]:
workflow = StateGraph(State)

tools = [get_formatted_item_context]
tool_node = ToolNode(tools)

workflow.add_node("tool_node", tool_node)
workflow.add_node("agent_node", agent_node)
workflow.add_node("intent_router_node", intent_router_node)

workflow.add_edge(START, "intent_router_node")

workflow.add_conditional_edges(
    "intent_router_node",
    intent_router_conditional_edges,
    {
        "agent_node": "agent_node",
        "end": END
    }
)

workflow.add_conditional_edges(
    "agent_node",
    tool_router,
    {
        "tools": "tool_node",
        "end": END
    }
)

workflow.add_edge("tool_node", "agent_node")

graph = workflow.compile()

In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
initial_state = {
    "messages": [HumanMessage(content="Can I get a tablet for my kid, a watch for me, a laptop for my wife and a waterproof speaker?")]
}

In [ ]:
result = graph.invoke(initial_state)

In [ ]:
result

In [ ]:
print(result["answer"])

In [ ]:
initial_state_1 = {
    "messages": [HumanMessage(content="I like neza Portable Bluetooth Speaker, could you give me more detail information about it?")]
}

In [ ]:
result_1 = graph.invoke(initial_state_1)

In [ ]:
result_1

In [ ]:
print(result_1["answer"])

### Persistent State

In [ ]:
from langgraph.checkpoint.postgres import PostgresSaver

### Set up the database (done once)

In [ ]:
with PostgresSaver.from_conn_string(
    "postgresql://langgraph_user: langgraph_password@localhost:5433/langgraph_db"
) as checkpointer:

    checkpointer.setup()

### Multiturn Conversation

In [ ]:
initial_state = {
    "messages": [HumanMessage(content="Can I get a tablet for my kid, a watch for me, a laptop for my wife and a waterproof speaker?")]
}

config = {
    "configurable": {
        "thread_id": "00000000000002"
    }
}

with PostgresSaver.from_conn_string(
    "postgresql://langgraph_user: langgraph_password@localhost:5433/langgraph_db"
) as checkpointer:

    graph = workflow.compile(checkpointer=checkpointer)

    result_1 = graph.invoke(initial_state, config)

In [ ]:
result_1

In [ ]:
print(result_1["answer"])

In [ ]:
initial_state_1 = {
    "messages": [HumanMessage(content="Oh I love neza Portable Bluetooth Speaker, could you give me more detailed information about it?")]
}

config = {
    "configurable": {
        "thread_id": "00000000000002"
    }
}

with PostgresSaver.from_conn_string(
    "postgresql://langgraph_user: langgraph_password@localhost:5433/langgraph_db"
) as checkpointer:

    graph = workflow.compile(checkpointer=checkpointer)

    result_2 = graph.invoke(initial_state_1, config)

In [ ]:
result_2

In [ ]:
print(result_2["answer"])

In [ ]:
initial_state_2 = {
    "messages": [HumanMessage(content="What about ASUS ZenBook?")]
}

config = {
    "configurable": {
        "thread_id": "00000000000002"
    }
}

with PostgresSaver.from_conn_string(
    "postgresql://langgraph_user: langgraph_password@localhost:5433/langgraph_db"
) as checkpointer:

    graph = workflow.compile(checkpointer=checkpointer)

    result_3 = graph.invoke(initial_state_2, config)

In [ ]:
result_3

In [ ]:
print(result_3["answer"])